# 102 - Plantilla de Proyecto Integrador — **Soluciones**

Implementación de un flujo completo sintético (tabular) con guardado de artefactos.

In [ ]:

import os, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1) Datos (sintéticos si no se encuentra data/dataset.csv)
Path("data").mkdir(exist_ok=True)
DATA_PATH = Path("data/dataset.csv")
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    rng = np.random.default_rng(0)
    df = pd.DataFrame({
        "feature1": rng.normal(0,1,500),
        "feature2": rng.uniform(-1,1,500),
        "target": (rng.random(500) > 0.5).astype(int)
    })
df.sample(5)


In [ ]:

# 2) Preprocesado y split
X = df[["feature1","feature2"]]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("Scaler listo")


In [ ]:

# 3) Modelo + evaluación
clf = LogisticRegression(max_iter=200)
clf.fit(X_train_s, y_train)
pred = clf.predict(X_test_s)
acc = accuracy_score(y_test, pred)
print("Accuracy:", round(acc,4))
print(classification_report(y_test, pred))


In [ ]:

# 4) Exportación de artefactos
Path("artifacts").mkdir(exist_ok=True)
joblib.dump({"scaler": scaler, "model": clf}, "artifacts/model.joblib")
df.head(30).to_csv("artifacts/sample.csv", index=False)
with open("artifacts/metrics.json","w",encoding="utf-8") as f:
    json.dump({"accuracy": float(acc)}, f, indent=2)
print("Artefactos guardados en ./artifacts")
